# MCP and Agent Interoperability

> **The story.** The Model Context Protocol standardizes discovery and invocation between AI applications and external capability servers. Its value is reducing bespoke integration glue while keeping schemas and lifecycle visible.
>
> **Where you are.** OrderFlow hard-codes inventory and pricing functions into every agent. Adding a second pricing provider requires agent-core edits.
>
> **Notation.** $C$ is the client; $S$ is an MCP server; $K(S)$ is its discovered capability set; each JSON-RPC request carries correlation ID $id$ and method $m$.

## 0 - The Challenge

> **The mission:** discover and use a second pricing provider through configuration alone, while rejecting an incompatible schema before any financial action.

```mermaid
flowchart LR
    A["Agent core"] --> G1["Bespoke inventory glue"]
    A --> G2["Bespoke pricing glue"]
    G1 --> M["MCP discovery + schemas"]
    G2 --> M
    M --> S["Config-selected providers"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G1 fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G2 fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style M fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style S fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Setup: import the deterministic OrderFlow runtime -----------------------
from pathlib import Path
import json
import sys

TRACK_DIR = Path(__vsc_ipynb_file__).resolve().parents[1] if '__vsc_ipynb_file__' in globals() else Path.cwd().resolve().parent
if str(TRACK_DIR) not in sys.path:
    sys.path.insert(0, str(TRACK_DIR))
from dataclasses import dataclass
from typing import Any, Callable

from pydantic import BaseModel, Field, ValidationError
from shared import INVENTORY, SUPPLIER_QUOTES

print("Local protocol simulation ready; no network transport is required.")


## 1 - Failure First: N-by-M Glue Hides Drift

With two agents and three services, bespoke adapters already create six integration relationships. Schema changes surface only when a call fails at runtime.

```mermaid
flowchart TD
    A1["Intake agent"] --> I["Inventory API"]
    A1 --> P1["Price API v1"]
    A1 --> P2["Price API v2"]
    A2["Supplier agent"] --> I
    A2 --> P1
    A2 --> P2
    style A1 fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style A2 fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P1 fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P2 fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Measure bespoke integration growth ----------------------------------
def integration_edges(agent_count, service_count):
    return agent_count * service_count

baseline_edges = integration_edges(2, 3)
future_edges = integration_edges(4, 8)
print(f"Bespoke edges now: {baseline_edges}; after growth: {future_edges}")
assert future_edges > baseline_edges * 5
print("Failure observed: adding agents and services multiplies glue code.")


## 2 - Build the Protocol Lifecycle

The smallest MCP-shaped lifecycle is initialize, discover, validate, call, and return a correlated result. Tools are one primitive; resources and prompts remain discoverable data, not executable authority.

```mermaid
sequenceDiagram
    participant C as Client
    participant S as Local MCP Server
    C->>S: initialize
    S-->>C: server info + protocol version
    C->>S: tools/list
    S-->>C: names + JSON schemas
    C->>S: tools/call(id, name, arguments)
    S-->>C: result(id) or typed error
```


In [ ]:
# -- Implement an in-process JSON-RPC server ------------------------------
@dataclass
class ToolSpec:
    name: str
    version: str
    input_schema: type[BaseModel]
    handler: Callable[..., dict[str, Any]]

class InventoryInput(BaseModel):
    sku: str

class PriceInputV1(BaseModel):
    sku: str

class LocalMCPServer:
    def __init__(self, name, protocol_version="2025-03-26"):
        self.name = name
        self.protocol_version = protocol_version
        self.tools = {}

    def register(self, spec):
        self.tools[spec.name] = spec

    def initialize(self):
        return {"server": self.name, "protocol_version": self.protocol_version, "capabilities": ["tools"]}

    def list_tools(self):
        return [{"name": spec.name, "version": spec.version, "schema": spec.input_schema.model_json_schema()} for spec in self.tools.values()]

    def call(self, request):
        request_id = request["id"]
        try:
            spec = self.tools[request["params"]["name"]]
            arguments = spec.input_schema.model_validate(request["params"]["arguments"])
            return {"jsonrpc": "2.0", "id": request_id, "result": spec.handler(**arguments.model_dump())}
        except (KeyError, ValidationError) as error:
            return {"jsonrpc": "2.0", "id": request_id, "error": {"code": -32602, "message": str(error)}}

inventory_server = LocalMCPServer("orderflow-inventory")
inventory_server.register(ToolSpec("inventory.lookup", "1.0.0", InventoryInput, lambda sku: {"sku": sku, "available": INVENTORY[sku]["available"]}))
print(inventory_server.initialize())
print(inventory_server.list_tools())


## 3 - Client Discovery, Validation, and Cancellation Boundaries

The client validates protocol version and input schema before calling. Unknown tools and incompatible argument shapes fail before a side effect.

```mermaid
flowchart LR
    D["Discover"] --> V{ "Compatible schema?" }
    V -->|"No"| F["Fail during validation"]
    V -->|"Yes"| C["Call with correlation ID"]
    C --> R["Result or cancellation"]
    style D fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style V fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Build a discovering client and prove schema failure -----------------
class MCPClient:
    def __init__(self, expected_protocol="2025-03-26"):
        self.expected_protocol = expected_protocol
        self.servers = {}
        self.catalog = {}

    def connect(self, alias, server):
        info = server.initialize()
        if info["protocol_version"] != self.expected_protocol:
            raise ValueError("incompatible_protocol")
        self.servers[alias] = server
        for tool in server.list_tools():
            self.catalog[f"{alias}:{tool['name']}"] = tool

    def call(self, alias, name, arguments, request_id):
        return self.servers[alias].call({"jsonrpc": "2.0", "id": request_id, "method": "tools/call", "params": {"name": name, "arguments": arguments}})

client = MCPClient()
client.connect("inventory", inventory_server)
ok = client.call("inventory", "inventory.lookup", {"sku": "SKU-CPU-01"}, "req-1")
bad = client.call("inventory", "inventory.lookup", {"product_code": "SKU-CPU-01"}, "req-2")
print("Valid:", ok)
print("Incompatible:", bad)
assert ok["result"]["available"] == 36
assert bad["error"]["code"] == -32602


## 4 - Add a Provider Through Configuration Alone

Agent-core code asks the client catalog for a configured pricing alias. The new provider implements the same versioned contract; the agent does not import provider-specific functions.

```mermaid
flowchart LR
    A["OrderFlow agent"] --> C["MCP client catalog"]
    C --> P1["primary pricing server"]
    C --> P2["backup pricing server"]
    CFG["configuration"] --> C
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P1 fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P2 fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style CFG fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Register two compatible providers and switch only configuration ------
def make_pricing_server(name, adjustment):
    server = LocalMCPServer(name)
    def price_lookup(sku):
        trusted = [quote for quote in SUPPLIER_QUOTES[sku] if quote["trusted"] and quote["age_hours"] <= 48]
        best = min(trusted, key=lambda item: item["unit_price"])
        return {"sku": sku, "supplier": best["supplier"], "unit_price": round(best["unit_price"] + adjustment, 2)}
    server.register(ToolSpec("pricing.quote", "1.0.0", PriceInputV1, price_lookup))
    return server

client.connect("primary", make_pricing_server("pricing-primary", 0.0))
client.connect("backup", make_pricing_server("pricing-backup", 3.0))

def agent_price_lookup(client, config, sku):
    alias = config["pricing_provider"]
    return client.call(alias, "pricing.quote", {"sku": sku}, f"price-{sku}")["result"]

primary = agent_price_lookup(client, {"pricing_provider": "primary"}, "SKU-CPU-01")
backup = agent_price_lookup(client, {"pricing_provider": "backup"}, "SKU-CPU-01")
print("Primary:", primary)
print("Backup:", backup)
assert primary["unit_price"] != backup["unit_price"]
print("PASS: provider changed through configuration without changing agent_price_lookup.")


## Roadmap Checkpoint

```mermaid
flowchart LR
    A["Protocol boundary met"] --> B["Next: multi-agent coordination"]
    style A fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

| Constraint | Before | After |
|---|---:|---:|
| Integration edges | N agents × M services | One client lifecycle per server |
| Capability discovery | Hard-coded imports | Runtime catalog |
| Schema drift | Runtime surprise | Validation error before action |
| New pricing provider | Agent edit | Configuration-only switch |

### Coverage Ledger

| Tier | Covered here |
|---|---|
| Built and measured | Lifecycle, tool discovery, JSON-RPC correlation, schema validation, provider switch |
| Explained and illustrated | Resources, prompts, cancellation, transports, authorization |
| Named with a reason | Remote HTTP transport and official SDK, deferred to keep the core offline |

### Key Takeaways

- Protocols reduce integration glue; they do not remove validation.
- Discovery is useful only when schemas and versions are checked.
- Correlation IDs make concurrent calls attributable.
- Use a normal function call when process or ownership boundaries do not exist.
